In [1]:
import os
import json
import pandas as pd
from tqdm import tqdm

In [12]:
from sort import Sort

In [2]:
rtdetr_kbrs_win1_275_path = "/mnt/nas/baecm/starcraft-vision/predictions/rtdetr_kbrs_win1_fold1_s123_kbrs025_20260615_045121/model_030/275.rep/all_correct.json"
rtdetr_kbrs_win4_275_path = "/mnt/nas/baecm/starcraft-vision/predictions/rtdetr_kbrs_win4_fold1_s123_kbrs025_20260615_045128/model_030/275.rep/all_correct.json"

maskrcnn_kbrs_win4_275_path = "/mnt/nas/baecm/starcraft-vision/predictions/maskrcnn_win4_kbrs_fold1_s123_kbrs025_base_score_20251219_080334/model_030/275.rep/all_correct.json"
maskrcnn_vanilla_win4_275_path = "/mnt/nas/baecm/starcraft-vision/predictions/maskrcnn_win4_vanilla_fold1_s123_20251201_072032/model_030/275.rep/all_correct.json"

In [3]:
with open(rtdetr_kbrs_win1_275_path, "r") as f:
    rkw1 = json.load(f)

In [4]:
with open(rtdetr_kbrs_win4_275_path, "r") as f:
    rkw4 = json.load(f)

In [5]:
with open(maskrcnn_kbrs_win4_275_path, "r") as f:
    mkw4 = json.load(f)

In [6]:
with open(maskrcnn_vanilla_win4_275_path, "r") as f:
    mvw4 = json.load(f)

In [13]:
def load_coco_and_prepare(coco_input):
    """COCO JSON 데이터를 읽어 Tracking에 적합한 DataFrame으로 정제합니다."""
    if isinstance(coco_input, str):
        with open(coco_input, 'r') as f:
            coco_data = json.load(f)
    elif isinstance(coco_input, dict):
        coco_data = coco_input
    else:
        raise ValueError("입력값은 파일 경로(str) 또는 딕셔너리(dict)여야 합니다.")
        
    df = pd.DataFrame(coco_data['annotations'])
    if df.empty:
        return df

    # [x_min, y_min, w, h] 포맷을 [x1, y1, x2, y2]로 변환
    bbox_df = pd.DataFrame(df['bbox'].tolist(), columns=['x', 'y', 'w', 'h'])
    df['x1'] = bbox_df['x']
    df['y1'] = bbox_df['y']
    df['x2'] = bbox_df['x'] + bbox_df['w']
    df['y2'] = bbox_df['y'] + bbox_df['h']
    
    return df

In [14]:
def get_spatially_distributed_top_k(frame_data, min_distance=30.0, top_k=5):
    """
    공간 클러스터링(Spatial NMS):
    가장 점수가 높은 객체를 선발하고, 지정된 반경(min_distance) 내의 다른 객체는 무시하여
    물리적으로 겹치지 않는 Top-K 지역 대표를 추출합니다.
    """
    # 중심점 계산
    cx = (frame_data['x1'] + frame_data['x2']) / 2.0
    cy = (frame_data['y1'] + frame_data['y2']) / 2.0
    frame_data = frame_data.assign(cx=cx, cy=cy)
    
    # 점수 내림차순 정렬
    frame_data = frame_data.sort_values(by='score', ascending=False)
    
    keep_indices = []
    kept_centers = []
    
    for idx, row in frame_data.iterrows():
        is_too_close = False
        
        # 선발된 대표 객체들과의 유클리드 거리 비교
        for kc in kept_centers:
            dist = np.sqrt((row['cx'] - kc[0])**2 + (row['cy'] - kc[1])**2)
            if dist < min_distance:
                is_too_close = True
                break
                
        # 충분히 멀리 떨어져 있다면 합격
        if not is_too_close:
            keep_indices.append(idx)
            kept_centers.append((row['cx'], row['cy']))
            
        # 목표 K개를 채우면 즉시 종료
        if top_k is not None and len(keep_indices) >= top_k:
            break
            
    return frame_data.loc[keep_indices]

In [15]:
def simulate_tracking(df, top_k_dets=None, min_score_threshold=0.0, min_distance=30.0):
    """
    정제된 프레임 데이터를 순회하며 공간 분산 필터링 후 SORT 추적을 수행합니다.
    """
    tracker = Sort(max_age=3, min_hits=3, iou_threshold=0.3)
    tracked_results = []
    frame_ids = sorted(df['image_id'].unique())
    
    for frame_id in tqdm(frame_ids, desc="Tracking Objects", unit="frame"):
        frame_data = df[df['image_id'] == frame_id]
        
        if not frame_data.empty:
            # 1. 최소 점수 미달 노이즈 컷
            if min_score_threshold > 0.0:
                frame_data = frame_data[frame_data['score'] >= min_score_threshold]
                
            # 2. 공간 분산이 보장된 Top-K 지역 대표 추출
            if top_k_dets is not None and not frame_data.empty:
                frame_data = get_spatially_distributed_top_k(
                    frame_data, 
                    min_distance=min_distance, 
                    top_k=top_k_dets
                )
                
        # 필터링 후 남은 객체가 없으면 칼만 필터 빈 업데이트 처리
        if frame_data.empty:
            tracker.update(np.empty((0, 5))) 
            continue

        dets = frame_data[['x1', 'y1', 'x2', 'y2', 'score']].values
        tracks = tracker.update(dets)
        
        for track in tracks:
            tracked_results.append({
                'image_id': frame_id,
                'track_id': int(track[4]),
                'bbox_xyxy': [track[0], track[1], track[2], track[3]]
            })
            
    return pd.DataFrame(tracked_results)

In [35]:
from tqdm import tqdm
import pandas as pd
import numpy as np
from sort import Sort

def get_spatially_distributed_top_k(frame_data, min_distance=30.0, top_k=5):
    """
    공간 클러스터링(Spatial NMS):
    가장 점수가 높은 객체를 선발하고, 지정된 반경(min_distance) 내의 다른 객체는 무시하여
    물리적으로 겹치지 않는 Top-K 지역 대표를 추출합니다.
    """
    cx = (frame_data['x1'] + frame_data['x2']) / 2.0
    cy = (frame_data['y1'] + frame_data['y2']) / 2.0
    frame_data = frame_data.assign(cx=cx, cy=cy)
    
    frame_data = frame_data.sort_values(by='score', ascending=False)
    
    keep_indices = []
    kept_centers = []
    
    for idx, row in frame_data.iterrows():
        is_too_close = False
        
        for kc in kept_centers:
            dist = np.sqrt((row['cx'] - kc[0])**2 + (row['cy'] - kc[1])**2)
            if dist < min_distance:
                is_too_close = True
                break
                
        if not is_too_close:
            keep_indices.append(idx)
            kept_centers.append((row['cx'], row['cy']))
            
        if top_k is not None and len(keep_indices) >= top_k:
            break
            
    return frame_data.loc[keep_indices]


def simulate_tracking_multi(df, top_k_dets=3, min_score_threshold=0.0, min_distance=30.0):
    """
    한 번의 루프(Single Pass)로 다음 4가지 결과를 동시에 산출합니다.
    (Tracked Top-K에는 공간 분산 클러스터링이 적용됩니다.)
    """
    tracker_all = Sort(max_age=3, min_hits=3, iou_threshold=0.3)
    tracker_topk = Sort(max_age=3, min_hits=3, iou_threshold=0.3)
     n
    results_raw_all = []
    results_raw_top1 = []
    results_tracked_all = []
    results_tracked_topk = []
    
    frame_ids = sorted(df['image_id'].unique())
    
    for frame_id in tqdm(frame_ids, desc="Processing All Pipelines", unit="frame"):
        frame_data = df[df['image_id'] == frame_id]
        
        if min_score_threshold > 0.0 and not frame_data.empty:
            frame_data = frame_data[frame_data['score'] >= min_score_threshold]
            
        if frame_data.empty:
            tracker_all.update(np.empty((0, 5)))
            tracker_topk.update(np.empty((0, 5)))
            continue

        frame_data = frame_data.sort_values(by='score', ascending=False)

        # ==========================================
        # [A & B] Raw 탐지 결과 저장
        # ==========================================
        for _, row in frame_data.iterrows():
            results_raw_all.append({
                'image_id': frame_id,
                'bbox_xyxy': [row['x1'], row['y1'], row['x2'], row['y2']],
                'score': row['score'] 
            })
            
        top1_row = frame_data.iloc[0]
        results_raw_top1.append({
            'image_id': frame_id,
            'bbox_xyxy': [top1_row['x1'], top1_row['y1'], top1_row['x2'], top1_row['y2']],
            'score': top1_row['score']
        })

        # ==========================================
        # [C] 전체 탐지 (All) 추적 실행
        # ==========================================
        dets_all = frame_data[['x1', 'y1', 'x2', 'y2', 'score']].values
        tracks_all = tracker_all.update(dets_all)
        
        for track in tracks_all:
            results_tracked_all.append({
                'image_id': frame_id,
                'track_id': int(track[4]),
                'bbox_xyxy': [track[0], track[1], track[2], track[3]]
            })

        # ==========================================
        # [D] Top-K 탐지 추적 실행 (★ 공간 분산 로직 적용 ★)
        # ==========================================
        # 단순 .head()가 아니라, 거리가 떨어진 3곳(top_k)의 대표를 뽑습니다.
        topk_data = get_spatially_distributed_top_k(
            frame_data, 
            min_distance=min_distance, 
            top_k=top_k_dets
        )
        
        dets_topk = topk_data[['x1', 'y1', 'x2', 'y2', 'score']].values
        tracks_topk = tracker_topk.update(dets_topk)
        
        for track in tracks_topk:
            results_tracked_topk.append({
                'image_id': frame_id,
                'track_id': int(track[4]),
                'bbox_xyxy': [track[0], track[1], track[2], track[3]]
            })

    return (
        pd.DataFrame(results_raw_all), 
        pd.DataFrame(results_raw_top1), 
        pd.DataFrame(results_tracked_all), 
        pd.DataFrame(results_tracked_topk)
    )

In [36]:
_ = load_coco_and_prepare(mkw4)
mkw4_all, mkw4_top1, mkw4_sort_all, mkw4_sort_top3 = simulate_tracking_multi(_, top_k_dets=3)

Processing All Pipelines: 100%|█████████████████████████| 66679/66679 [06:31<00:00, 170.21frame/s]


In [37]:
_ = load_coco_and_prepare(mvw4)
mvw4_all, mvw4_top1, mvw4_sort_all, mvw4_sort_top3 = simulate_tracking_multi(_, top_k_dets=3)

Processing All Pipelines: 100%|█████████████████████████| 66673/66673 [06:38<00:00, 167.36frame/s]


In [38]:
_ = load_coco_and_prepare(rkw4)
rkw4_all, rkw4_top1, rkw4_sort_all, rkw4_sort_top3 = simulate_tracking_multi(_, top_k_dets=3)

Processing All Pipelines: 100%|█████████████████████████| 66667/66667 [06:24<00:00, 173.47frame/s]


In [39]:
def select_sequence_range(df, start, finish):
    return df.iloc[df["image_id"] >= start].iloc[df["image_id"] < finish]

In [40]:
mkw4_all_seq = select_sequence_range(mkw4_all, 30000, 32400)
mkw4_top1_seq = select_sequence_range(mkw4_top1, 30000, 32400)
mkw4_sort_all_seq = select_sequence_range(mkw4_sort_all, 30000, 32400)
mkw4_sort_top3_seq = select_sequence_range(mkw4_sort_top3, 30000, 32400)

mvw4_all_seq = select_sequence_range(mvw4_all, 30000, 32400) 
mvw4_top1_seq = select_sequence_range(mvw4_top1, 30000, 32400) 
mvw4_sort_all_seq = select_sequence_range(mvw4_sort_all, 30000, 32400) 
mvw4_sort_top3_seq = select_sequence_range(mvw4_sort_top3, 30000, 32400) 

rkw4_all_seq = select_sequence_range(rkw4_all, 30000, 32400)
rkw4_top1_seq = select_sequence_range(rkw4_top1, 30000, 32400)
rkw4_sort_all_seq = select_sequence_range(rkw4_sort_all, 30000, 32400)
rkw4_sort_top3_seq = select_sequence_range(rkw4_sort_top3, 30000, 32400)

In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from tqdm import tqdm

def adjust_bbox_for_pyplot(bbox, fixed_w=20, fixed_h=12):
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    x_min = cx - (fixed_w / 2.0)
    y_min = cy - (fixed_h / 2.0)
    return x_min, y_min, fixed_w, fixed_h

def create_tracking_animation(tracking_df, output_file="tracking_output", grid_size=128, fps=24, save_mp4=True, save_gif=False, title="Tracking Process"):
    fig, ax = plt.subplots(figsize=(6, 6))
    frame_ids = sorted(tracking_df['image_id'].unique())
    
    cmap = plt.get_cmap('tab20')
    color_map = {}
    
    def update(frame_index):
        ax.clear()
        frame_id = frame_ids[frame_index]
        
        ax.set_xlim(0, grid_size)
        ax.set_ylim(grid_size, 0) 
        ax.set_title(f"{title} - Frame: {frame_id}")
        
        ax.set_xticks(np.arange(0, grid_size+1, 16))
        ax.set_yticks(np.arange(0, grid_size+1, 16))
        ax.grid(True, linestyle='--', alpha=0.5)
        
        current_data = tracking_df[tracking_df['image_id'] == frame_id]
        
        has_track_id = 'track_id' in current_data.columns
        
        # [핵심 수정 부분] 
        local_idx = 0 # Raw 데이터용 프레임 내 순위 인덱스
        
        for _, row in current_data.iterrows():
            if has_track_id:
                # 1. 추적 데이터(Tracked)인 경우: 고유 UID를 그대로 사용 (색상 일관성 유지)
                uid = int(row['track_id'])
                label_text = f"ID: {uid}"
            else:
                # 2. 원본 데이터(Raw)인 경우: 현재 프레임 내의 '점수 순위'를 임시 ID로 사용
                # (가장 점수가 높은 객체는 항상 같은 색으로 표시됨)
                uid = local_idx 
                label_text = f"Rank: {uid + 1}"
                local_idx += 1
            
            original_bbox = row['bbox_xyxy']
            x_min, y_min, w, h = adjust_bbox_for_pyplot(original_bbox, fixed_w=20, fixed_h=12)
            
            # 색상 매핑 (딕셔너리에 없으면 새로운 색상 할당)
            if uid not in color_map:
                color_map[uid] = cmap(len(color_map) % 20)
            color = color_map[uid]
            
            # 바운딩 박스 렌더링
            rect = patches.Rectangle((x_min, y_min), w, h, 
                                     linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            
            # 텍스트 렌더링
            ax.text(x_min, y_min - 2, label_text, color=color, 
                    fontsize=10, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))

    ani = FuncAnimation(fig, update, frames=len(frame_ids), interval=1000/fps)
    
    print(f"총 {len(frame_ids)}개의 프레임 렌더링을 시작합니다... [{title}]")

    if save_mp4:
        with tqdm(total=len(frame_ids), desc="Rendering MP4", unit="frame") as pbar:
            def update_progress_mp4(current_frame, total_frames):
                pbar.update(1)
            ani.save(f"{output_file}.mp4", writer='ffmpeg', fps=fps, progress_callback=update_progress_mp4)
            
    if save_gif:
        with tqdm(total=len(frame_ids), desc="Rendering GIF", unit="frame") as pbar:
            def update_progress_gif(current_frame, total_frames):
                pbar.update(1)
            ani.save(f"{output_file}.gif", writer='pillow', fps=fps, progress_callback=update_progress_gif)
    
    print(f"렌더링 완료! 파일이 저장되었습니다: {output_file}")
    plt.close(fig)

In [34]:
# from tqdm import tqdm
# import pandas as pd
# import numpy as np
# from sort import Sort

# def get_spatially_distributed_top_k(frame_data, min_distance=30.0, top_k=5):
#     """
#     점수가 높은 순으로 탐색하되, 이미 선택된 '지역 대표'와 
#     min_distance 이내로 가까운 위치에 있는 객체는 무시합니다.
#     """
#     # 1. 중심점 계산 (그룹화의 기준)
#     cx = (frame_data['x1'] + frame_data['x2']) / 2.0
#     cy = (frame_data['y1'] + frame_data['y2']) / 2.0
#     frame_data = frame_data.assign(cx=cx, cy=cy)
    
#     # 2. Score 기준으로 내림차순 정렬 (가장 확신하는 객체부터 탐색)
#     frame_data = frame_data.sort_values(by='score', ascending=False)
    
#     keep_indices = []
#     kept_centers = []
    
#     for idx, row in frame_data.iterrows():
#         is_too_close = False
        
#         # 3. 이미 선발된 지역 대표들과의 거리 측정
#         for kc in kept_centers:
#             dist = np.sqrt((row['cx'] - kc[0])**2 + (row['cy'] - kc[1])**2)
#             if dist < min_distance:
#                 is_too_close = True
#                 break  # 하나라도 너무 가까우면 이 객체는 탈락
                
#         # 4. 충분히 멀리 떨어져 있다면 새로운 지역 대표로 선발
#         if not is_too_close:
#             keep_indices.append(idx)
#             kept_centers.append((row['cx'], row['cy']))
            
#         # 5. 원하는 K개를 모두 찾았다면 조기 종료
#         if top_k is not None and len(keep_indices) >= top_k:
#             break
            
#     # 선발된 인덱스만 필터링하여 반환
#     return frame_data.loc[keep_indices]


# def simulate_tracking(df, top_k_dets=None, min_score_threshold=0.0, min_distance=30.0):
#     tracker = Sort(max_age=3, min_hits=3, iou_threshold=0.3)
#     tracked_results = []
#     frame_ids = sorted(df['image_id'].unique())
    
#     for frame_id in tqdm(frame_ids, desc="Tracking Objects", unit="frame"):
#         frame_data = df[df['image_id'] == frame_id]
        
#         if frame_data.empty:
#             tracker.update(np.empty((0, 5))) 
#             continue
            
#         # 1. 최소 점수 미달 노이즈 컷
#         if min_score_threshold > 0.0:
#             frame_data = frame_data[frame_data['score'] >= min_score_threshold]
            
#         # =======================================================
#         # ★ [수정된 로직] 단순 Top-K가 아닌 '공간적 분산이 보장된' Top-K 추출
#         # =======================================================
#         if top_k_dets is not None and not frame_data.empty:
#             frame_data = get_spatially_distributed_top_k(
#                 frame_data, 
#                 min_distance=min_distance, # 이 반경 내에서는 1등 하나만 살아남음
#                 top_k=top_k_dets
#             )
#         # =======================================================
        
#         if frame_data.empty:
#             tracker.update(np.empty((0, 5))) 
#             continue

#         dets = frame_data[['x1', 'y1', 'x2', 'y2', 'score']].values
#         tracks = tracker.update(dets)
        
#         for track in tracks:
#             tracked_results.append({
#                 'image_id': frame_id,
#                 'track_id': int(track[4]),
#                 'bbox_xyxy': [track[0], track[1], track[2], track[3]]
#             })
            
#     return pd.DataFrame(tracked_results)

# # --- 실행 예시 ---
# # 반경 30 픽셀 내에서는 가장 점수가 높은 1개만 남기고, 화면 전체에서 최대 5개의 독립된 구역을 추적
# # tracking_result_df = simulate_tracking(coco_df, top_k_dets=5, min_score_threshold=0.3, min_distance=30.0)

In [48]:
create_tracking_animation(mvw4_all_seq, output_file="mvw4_all_seq", title="Mask RCNN", grid_size=128, fps=24)

총 2400개의 프레임 렌더링을 시작합니다... [Mask RCNN]


Rendering GIF: 100%|███████████████████████████████████████| 2400/2400 [06:23<00:00,  6.25frame/s]

렌더링 완료! 파일이 저장되었습니다: mvw4_all_seq


In [55]:
create_tracking_animation(mkw4_all_seq, output_file="mkw4_all_seq", title="Mask RCNN + Saliency Guide", grid_size=128, fps=24)

총 2400개의 프레임 렌더링을 시작합니다... [Mask RCNN + Saliency Guide]


Rendering MP4: 100%|███████████████████████████████████████| 2400/2400 [04:22<00:00,  9.13frame/s]

렌더링 완료! 파일이 저장되었습니다: mkw4_all_seq


In [56]:
create_tracking_animation(mkw4_sort_all_seq, output_file="mkw4_sort_all_seq", title="Mask RCNN + Saliency Guide + Post-processing (all)", grid_size=128, fps=24)

총 2400개의 프레임 렌더링을 시작합니다... [Mask RCNN + Saliency Guide + Post-processing (all)]


Rendering MP4: 100%|███████████████████████████████████████| 2400/2400 [04:20<00:00,  9.22frame/s]

렌더링 완료! 파일이 저장되었습니다: mkw4_sort_all_seq


In [57]:
create_tracking_animation(mkw4_sort_top3_seq, output_file="mkw4_sort_top3_seq", title="Mask RCNN + Saliency Guide + Post-processing (top-3)", grid_size=128, fps=24)

총 2400개의 프레임 렌더링을 시작합니다... [Mask RCNN + Saliency Guide + Post-processing (top-3)]


Rendering MP4: 100%|███████████████████████████████████████| 2400/2400 [03:54<00:00, 10.22frame/s]

렌더링 완료! 파일이 저장되었습니다: mkw4_sort_top3_seq


In [65]:
create_tracking_animation(rkw4_all_seq, output_file="rkw4_all_seq", title="RT-DETR + Saliency (all)", grid_size=128, fps=24)

총 2400개의 프레임 렌더링을 시작합니다... [RT-DETR + Saliency (all)]


Rendering MP4: 100%|███████████████████████████████████████| 2400/2400 [04:07<00:00,  9.70frame/s]

렌더링 완료! 파일이 저장되었습니다: rkw4_all_seq


In [62]:
create_tracking_animation(rkw4_top1_seq, output_file="rkw4_top1_seq", title="RT-DETR + Saliency (top-1)", grid_size=128, fps=24)

총 2400개의 프레임 렌더링을 시작합니다... [RT-DETR + Saliency (top-1)]


Rendering MP4: 100%|███████████████████████████████████████| 2400/2400 [03:39<00:00, 10.94frame/s]

렌더링 완료! 파일이 저장되었습니다: rkw4_top1_seq


In [63]:
create_tracking_animation(rkw4_sort_all_seq, output_file="rkw4_sort_all_seq", title="RT-DETR + Saliency + Post-processing (all)", grid_size=128, fps=24)

총 2385개의 프레임 렌더링을 시작합니다... [RT-DETR + Saliency + Post-processing (all)]


Rendering MP4: 100%|███████████████████████████████████████| 2385/2385 [04:01<00:00,  9.87frame/s]

렌더링 완료! 파일이 저장되었습니다: rkw4_sort_all_seq


In [64]:
create_tracking_animation(rkw4_sort_top3_seq, output_file="rkw4_sort_top3_seq", title="RT-DETR + Saliency + Post-processing (top-3)", grid_size=128, fps=24)

총 2287개의 프레임 렌더링을 시작합니다... [RT-DETR + Saliency + Post-processing (top-3)]


Rendering MP4: 100%|███████████████████████████████████████| 2287/2287 [03:41<00:00, 10.32frame/s]

렌더링 완료! 파일이 저장되었습니다: rkw4_sort_top3_seq


In [109]:
import pandas as pd
import numpy as np

def analyze_track_consistency(tracking_df, fps=24, jump_threshold=50.0, gap_tolerance=3):
    """
    정답(GT) 없이 DataFrame 자체의 물리적 타당성을 기반으로 추적 일관성을 평가합니다.
    - jump_threshold: 1프레임당 이동할 수 있는 최대 허용 픽셀 거리
    - gap_tolerance: 추적이 끊겨도 같은 객체로 인정해 주는 최대 누락 프레임 수
    """
    df = tracking_df.copy()
    
    # 1. 중심점 좌표(cx, cy) 계산
    bbox = pd.DataFrame(df['bbox_xyxy'].tolist(), columns=['x1', 'y1', 'x2', 'y2'])
    df['cx'] = (bbox['x1'] + bbox['x2']) / 2.0
    df['cy'] = (bbox['y1'] + bbox['y2']) / 2.0
    
    # 프레임 순서대로 정렬
    df = df.sort_values(by=['track_id', 'image_id'])
    
    # 이전 프레임 데이터 및 프레임 차이(gap) 계산
    df['prev_frame'] = df.groupby('track_id')['image_id'].shift(1)
    df['prev_cx'] = df.groupby('track_id')['cx'].shift(1)
    df['prev_cy'] = df.groupby('track_id')['cy'].shift(1)
    
    # 프레임 간격(Delta Frame) 및 픽셀 이동 거리 계산
    df['frame_diff'] = df['image_id'] - df['prev_frame']
    df['pixel_jump'] = np.sqrt((df['cx'] - df['prev_cx'])**2 + (df['cy'] - df['prev_cy'])**2)
    
    # ========================================================
    # ★ 핵심 로직: 프레임 간격에 따른 상태 분류
    # ========================================================
    
    # Case 1: 연속 탐지 중 비정상 도약 (1프레임 만에 임계값 초과)
    df['switch_consecutive'] = (df['frame_diff'] == 1) & (df['pixel_jump'] > jump_threshold)
    
    # Case 2: 허용 범위 내 누락 후 복귀 성공 (정상적인 Occlusion 처리)
    # 누락된 프레임 수만큼 허용 거리도 비례해서 늘려줍니다. (선형 등속 운동 가정)
    max_allowed_jump = jump_threshold * df['frame_diff']
    df['valid_occlusion'] = (df['frame_diff'] > 1) & (df['frame_diff'] <= gap_tolerance) & (df['pixel_jump'] <= max_allowed_jump)
    
    # Case 3: 허용 범위 내 누락이 있었으나, 엉뚱한 위치에서 나타남 (가려진 사이 다른 객체로 ID 튐)
    df['switch_gap'] = (df['frame_diff'] > 1) & (df['frame_diff'] <= gap_tolerance) & (df['pixel_jump'] > max_allowed_jump)
    
    # Case 4: 허용 범위를 초과하여 너무 오랜만에 과거 ID가 부활함 (Tracker 오류)
    df['switch_over_tolerance'] = df['frame_diff'] > gap_tolerance

    # ========================================================
    
    # Track ID별로 통계 집계
    track_stats = df.groupby('track_id').agg(
        frame_count=('image_id', 'count'),
        first_frame=('image_id', 'min'),
        last_frame=('image_id', 'max'),
        valid_occlusions=('valid_occlusion', 'sum'),
        err_consecutive=('switch_consecutive', 'sum'),
        err_gap=('switch_gap', 'sum'),
        err_over_tolerance=('switch_over_tolerance', 'sum')
    )
    
    track_stats['lifespan'] = track_stats['last_frame'] - track_stats['first_frame'] + 1
    
    # 총 에러(비정상 도약) 횟수 합산
    track_stats['total_id_switches'] = track_stats['err_consecutive'] + track_stats['err_gap'] + track_stats['err_over_tolerance']
    
    # 불리언(True/False) 합산을 정수형으로 변환
    cols_to_int = ['valid_occlusions', 'err_consecutive', 'err_gap', 'err_over_tolerance', 'total_id_switches']
    track_stats[cols_to_int] = track_stats[cols_to_int].astype(int)
    
    # --- 전체 요약 리포트 출력 ---
    print("========== 추적 일관성 정밀 진단 리포트 ==========")
    print(f"분석 대상 고유 객체(UID) 수: {len(track_stats)}")
    print(f"평균 추적 수명: {track_stats['lifespan'].mean():.1f} 프레임")
    
    short_tracks = len(track_stats[track_stats['lifespan'] < (fps * 1.0)])
    print(f"1초({fps}프레임) 미만 단명 객체 비율: {short_tracks/len(track_stats)*100:.1f}%")
    print("-" * 50)
    print(f"✅ 정상적인 가림(Occlusion) 극복 횟수: {track_stats['valid_occlusions'].sum()} 회")
    print(f"❌ 총 ID Switch 의심 횟수: {track_stats['total_id_switches'].sum()} 회")
    print(f"   ├─ 연속 프레임 도약 (단순 추적 튐): {track_stats['err_consecutive'].sum()} 회")
    print(f"   ├─ 짧은 가림 중 도약 (다른 객체로 옮겨감): {track_stats['err_gap'].sum()} 회")
    print(f"   └─ 긴 공백 후 부활 (과거 ID 잘못 꺼냄): {track_stats['err_over_tolerance'].sum()} 회")
    print("==================================================\n")
    
    return track_stats, df

# --- 실행 예시 ---
# gap_tolerance는 SORT의 'max_age' 파라미터 값과 동일하거나 약간 크게 설정하는 것이 좋습니다.
# stats_df, detailed_df = analyze_track_consistency(b, fps=24, jump_threshold=30.0, gap_tolerance=3)

In [110]:
# 실행 예시
rkw1_track_stats_df, rkw1_detailed_df = analyze_track_consistency(rkw1_tracked, fps=24, jump_threshold=30.0, gap_tolerance=5)
rkw4_track_stats_df, rkw4_detailed_df = analyze_track_consistency(rkw4_tracked, fps=24, jump_threshold=30.0, gap_tolerance=5)
mkw4_track_stats_df, mkw4_detailed_df = analyze_track_consistency(mkw4_tracked, fps=24, jump_threshold=30.0, gap_tolerance=5)
mvw4_track_stats_df, mvw4_detailed_df = analyze_track_consistency(mvw4_tracked, fps=24, jump_threshold=30.0, gap_tolerance=5)

========== 추적 일관성 정밀 진단 리포트 ==========
분석 대상 고유 객체(UID) 수: 3317
평균 추적 수명: 39.6 프레임
1초(24프레임) 미만 단명 객체 비율: 66.7%
--------------------------------------------------
✅ 정상적인 가림(Occlusion) 극복 횟수: 1434 회
❌ 총 ID Switch 의심 횟수: 733 회
   ├─ 연속 프레임 도약 (단순 추적 튐): 0 회
   ├─ 짧은 가림 중 도약 (다른 객체로 옮겨감): 0 회
   └─ 긴 공백 후 부활 (과거 ID 잘못 꺼냄): 733 회

========== 추적 일관성 정밀 진단 리포트 ==========
분석 대상 고유 객체(UID) 수: 3712
평균 추적 수명: 38.1 프레임
1초(24프레임) 미만 단명 객체 비율: 66.9%
--------------------------------------------------
✅ 정상적인 가림(Occlusion) 극복 횟수: 1768 회
❌ 총 ID Switch 의심 횟수: 906 회
   ├─ 연속 프레임 도약 (단순 추적 튐): 0 회
   ├─ 짧은 가림 중 도약 (다른 객체로 옮겨감): 0 회
   └─ 긴 공백 후 부활 (과거 ID 잘못 꺼냄): 906 회

========== 추적 일관성 정밀 진단 리포트 ==========
분석 대상 고유 객체(UID) 수: 1681
평균 추적 수명: 102.4 프레임
1초(24프레임) 미만 단명 객체 비율: 60.4%
--------------------------------------------------
✅ 정상적인 가림(Occlusion) 극복 횟수: 773 회
❌ 총 ID Switch 의심 횟수: 354 회
   ├─ 연속 프레임 도약 (단순 추적 튐): 0 회
   ├─ 짧은 가림 중 도약 (다른 객체로 옮겨감): 0 회
   └─ 긴 공백 후 부활 (과거 ID 잘못 꺼냄): 354 회

========== 추적

In [105]:
import numpy as np
import pandas as pd
from tqdm import tqdm

def analyze_spatial_coverage(tracking_df, grid_size=128):
    """
    128x128 그리드 내에서 매 프레임별 바운딩 박스들이 
    실제로 커버하는 순수 면적(합집합)과 비율을 계산합니다.
    """
    df = tracking_df.copy()
    
    # 결과를 담을 리스트
    coverage_results = []
    
    # 프레임 오름차순 정렬
    frame_ids = sorted(df['image_id'].unique())
    total_area = grid_size * grid_size # 128 * 128 = 16384
    
    for frame_id in tqdm(frame_ids, desc="Calculating Coverage", unit="frame"):
        frame_data = df[df['image_id'] == frame_id]
        
        # 1. 128x128 크기의 False로 채워진 빈 캔버스(Mask) 생성
        mask = np.zeros((grid_size, grid_size), dtype=bool)
        
        for _, row in frame_data.iterrows():
            # 박스 좌표 추출 (앞서 20x12로 고정한 좌표를 사용하셔도 됩니다)
            x1, y1, x2, y2 = row['bbox_xyxy']
            
            # 2. 캔버스 밖으로 나가는 좌표를 0 ~ 128 사이로 잘라냄(Clipping)
            # (박스가 화면 모서리에 걸쳐 있을 때 인덱스 에러 방지)
            x1_clip, x2_clip = np.clip([int(x1), int(x2)], 0, grid_size)
            y1_clip, y2_clip = np.clip([int(y1), int(y2)], 0, grid_size)
            
            # 3. 박스 영역을 True로 활성화 (겹치는 부분은 알아서 덮어씌워짐)
            # 주의: numpy 배열 슬라이싱은 [y축, x축] 순서입니다.
            mask[y1_clip:y2_clip, x1_clip:x2_clip] = True
            
        # 4. 활성화된(True) 픽셀의 총합 = 실제 커버된 면적
        covered_pixels = np.sum(mask)
        coverage_ratio = (covered_pixels / total_area) * 100.0
        
        coverage_results.append({
            'image_id': frame_id,
            'object_count': len(frame_data),
            'covered_pixels': covered_pixels,
            'coverage_ratio_%': coverage_ratio
        })
        
    # DataFrame으로 변환
    coverage_df = pd.DataFrame(coverage_results)
    
    # --- 전체 요약 리포트 출력 ---
    mean_ratio = coverage_df['coverage_ratio_%'].mean()
    max_ratio = coverage_df['coverage_ratio_%'].max()
    max_frame = coverage_df.loc[coverage_df['coverage_ratio_%'].idxmax(), 'image_id']
    
    print("\n========== 공간 커버리지 진단 리포트 ==========")
    print(f"전체 화면 크기: {grid_size} x {grid_size} ({total_area} 픽셀)")
    print(f"평균 화면 커버리지: {mean_ratio:.2f}%")
    print(f"최대 화면 커버리지: {max_ratio:.2f}% (발생 프레임: {max_frame})")
    print("===============================================\n")
    
    return coverage_df

# --- 실행 예시 ---
# coverage_stats_df = analyze_spatial_coverage(b, grid_size=128)
# print(coverage_stats_df.head(10))

In [111]:
coverage_stats_rkw1 = analyze_spatial_coverage(rkw1_tracked, grid_size=128)
coverage_stats_rkw4 = analyze_spatial_coverage(rkw4_tracked, grid_size=128)
coverage_stats_mkw4 = analyze_spatial_coverage(mkw4_tracked, grid_size=128)
coverage_stats_mvw4 = analyze_spatial_coverage(mvw4_tracked, grid_size=128)

Calculating Coverage: 100%|████████████████████████████| 63860/63860 [00:42<00:00, 1502.17frame/s]



========== 공간 커버리지 진단 리포트 ==========
전체 화면 크기: 128 x 128 (16384 픽셀)
평균 화면 커버리지: 2.79%
최대 화면 커버리지: 5.08% (발생 프레임: 18180)



Calculating Coverage: 100%|████████████████████████████| 64531/64531 [00:44<00:00, 1454.12frame/s]



========== 공간 커버리지 진단 리포트 ==========
전체 화면 크기: 128 x 128 (16384 픽셀)
평균 화면 커버리지: 2.95%
최대 화면 커버리지: 5.08% (발생 프레임: 36412)



Calculating Coverage: 100%|████████████████████████████| 66567/66567 [00:50<00:00, 1317.66frame/s]



========== 공간 커버리지 진단 리포트 ==========
전체 화면 크기: 128 x 128 (16384 픽셀)
평균 화면 커버리지: 3.66%
최대 화면 커버리지: 5.05% (발생 프레임: 3081)



Calculating Coverage: 100%|████████████████████████████| 66474/66474 [00:50<00:00, 1309.75frame/s]



========== 공간 커버리지 진단 리포트 ==========
전체 화면 크기: 128 x 128 (16384 픽셀)
평균 화면 커버리지: 3.75%
최대 화면 커버리지: 5.04% (발생 프레임: 48539)



In [71]:
rkw4_track_stats_df.sort_values(by='suspected_id_switches', ascending=False)

,frame_count,first_frame,last_frame,lifespan,missing_frames,missing_ratio (%),suspected_id_switches
track_id,,,,,,,
20940,235,3,237,235,0,0.00,0
20941,106,3,108,106,0,0.00,0
20942,235,3,237,235,0,0.00,0
20943,237,3,239,237,0,0.00,0
20944,235,3,237,235,0,0.00,0
...,...,...,...,...,...,...,...
36733,11,66593,66603,11,0,0.00,0
36734,91,66594,66687,94,3,3.19,0
36735,2,66597,66598,2,0,0.00,0


In [72]:
mkw4_track_stats_df.sort_values(by='suspected_id_switches', ascending=False)

,frame_count,first_frame,last_frame,lifespan,missing_frames,missing_ratio (%),suspected_id_switches
track_id,,,,,,,
36754,4631,3,4655,4653,22,0.47,0
36755,4672,3,4677,4675,3,0.06,0
36756,2,238,239,2,0,0.00,0
36758,3,260,262,3,0,0.00,0
36759,226,302,527,226,0,0.00,0
...,...,...,...,...,...,...,...
47230,19,66669,66687,19,0,0.00,0
47232,10,66678,66687,10,0,0.00,0
47233,7,66681,66687,7,0,0.00,0


In [73]:
mvw4_track_stats_df.sort_values(by='suspected_id_switches', ascending=False)

,frame_count,first_frame,last_frame,lifespan,missing_frames,missing_ratio (%),suspected_id_switches
track_id,,,,,,,
47237,2241,3,2247,2245,4,0.18,0
47238,2514,3,2516,2514,0,0.00,0
47239,8,198,205,8,0,0.00,0
47240,113,241,353,113,0,0.00,0
47241,41,305,345,41,0,0.00,0
...,...,...,...,...,...,...,...
61218,14,66671,66687,17,3,17.65,0
61219,11,66668,66678,11,0,0.00,0
61220,7,66681,66687,7,0,0.00,0
